In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box


In [2]:
print(Path.cwd())

c:\Users\kaursa40847\project


In [ ]:
# Define file paths for monitoring data (main dataset for observation plots) and output directory
MONITORING = Path("original/detailed_monitoring_plots_2017_2026_est.csv")
OUTPUT = Path("working")


In [4]:

raw = pd.read_csv(MONITORING, sep=";")



In [5]:
raw.head()

,Programm,Seiretöö nimetus,Vastutav partner,Vastutav isik,Seirekoha KKR,Seirekoha nimi,Jahipiirkonna peamine maakond,Seirekoha staatus,Seirekoha x L-EST97,Seirekoha y L-EST97,...,Mõõtmata,Viimane kommentaar väärtuse juures,Analüüsimeetodi standard,Analüüsimeetodi nimetus,Analüüsimeetodi tööjuhendi nr,Analüüsimeetodi allikas,Meetodi täpsustus,Mullaproov,Mullahorisont,Mullaproovi sügavus
0,Ohustatud taimekoosluste (Natura elupaigad) seire,Metsaelupaikade seire 2017. a.,Anneli Palo FIE,Anneli Palo,NaN,"Harju maakond, Raasiku vald, Perila küla",NaN,Kehtiv,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Ohustatud taimekoosluste (Natura elupaigad) seire,Metsaelupaikade seire 2017. a.,Anneli Palo FIE,Anneli Palo,NaN,"Järva maakond, Paide linn, Esna küla; Järva ma...",NaN,Arhiveeritud,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Ohustatud taimekoosluste (Natura elupaigad) seire,Metsaelupaikade seire 2017. a.,Anneli Palo FIE,Anneli Palo,NaN,"Lääne-Viru maakond, Tapa vald, Kõrveküla küla",NaN,Kehtiv,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Ohustatud taimekoosluste (Natura elupaigad) seire,Metsaelupaikade seire 2017. a.,Anneli Palo FIE,Anneli Palo,NaN,"Lääne-Viru maakond, Tapa vald, Kõrveküla küla",NaN,Kehtiv,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Ohustatud taimekoosluste (Natura elupaigad) seire,Metsaelupaikade seire 2017. a.,Anneli Palo FIE,Anneli Palo,NaN,"Lääne-Viru maakond, Tapa vald, Kõrveküla küla",NaN,Kehtiv,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# In dataset, each plot has multiple rows (one row per measured indicator). Additionally some basic data cleaning steps are applied to ensure consistency.

raw = raw.loc[
    raw["Seiretöö nimetus"].str.startswith("Metsaelupaikade seire", na=False)
    & raw["Vaatlusgrupp"].str.contains(r"r\s*=\s*20m", na=False)
].dropna(subset=["Proovi / vaatluse kood"])

id_col, indicator, value = "Proovi / vaatluse kood", "Näitaja nimetus", "Väärtus (muud)"
meta = raw.groupby(id_col, as_index=False)[
    ["Sisestatud x L-EST97", "Sisestatud y L-EST97", "Seireaja algus"]
].first()
wide = raw.pivot_table(index=id_col, columns=indicator, values=value, aggfunc="first")
plots = meta.join(wide, on=id_col)


# The original dataset uses 'Sisestatud y' for EASTING and 'x' for NORTHING.
plots["x"] = pd.to_numeric(plots["Sisestatud y L-EST97"].str.replace(",", "."), errors="coerce")
plots["y"] = pd.to_numeric(plots["Sisestatud x L-EST97"].str.replace(",", "."), errors="coerce")
plots = plots.loc[plots["x"].between(300000, 800000) &
                  plots["y"].between(6300000, 6700000)].copy()
plots["habitat"] = plots["Elupaigatüüp (LD)"].astype("string")

# Contradictory answers for the same plot/indicator should NOT count as labels.
conflict = raw.groupby([id_col, indicator])[value].nunique(dropna=True)
# A plot with contradictory habitat answers is not usable as a habitat label.
invalid_ids = [plot_id for plot_id, question in conflict.loc[conflict > 1].index
               if question == "Elupaigatüüp (LD)"]
plots.loc[plots[id_col].isin(invalid_ids), "habitat"] = pd.NA

# Repeated visits to exactly the same coordinates are one location,
# not several geographically independent plot observations.
plots["survey_date"] = pd.to_datetime(plots["Seireaja algus"], errors="coerce",
                                      dayfirst=True, format="mixed")
plots = plots.sort_values("survey_date").drop_duplicates(["x", "y"], keep="last")
plots = gpd.GeoDataFrame(plots.copy(), geometry=gpd.points_from_xy(plots.x, plots.y), crs="EPSG:3301")

plots["source"] = "field_plot"
print("Unique monitoring locations:", len(plots))
print("Columns:", plots.columns.tolist())
plots["source"] = "field_plot"
print("Unique monitoring locations:", len(plots))

Unique monitoring locations: 1698
Columns: ['Proovi / vaatluse kood', 'Sisestatud x L-EST97', 'Sisestatud y L-EST97', 'Seireaja algus', 'Elupaiga esinduslikkus', 'Elupaiga funktsioonide säilimine', 'Elupaiga üldine looduskaitseline väärtus', 'Elupaigatüüp (LD)', 'Kasvukohatüüp (Paal 1997)', 'Koosluse struktuuri säilimine', 'x', 'y', 'habitat', 'survey_date', 'geometry', 'source']
Unique monitoring locations: 1698


In [7]:
plots.head()

,Proovi / vaatluse kood,Sisestatud x L-EST97,Sisestatud y L-EST97,Seireaja algus,Elupaiga esinduslikkus,Elupaiga funktsioonide säilimine,Elupaiga üldine looduskaitseline väärtus,Elupaigatüüp (LD),Kasvukohatüüp (Paal 1997),Koosluse struktuuri säilimine,x,y,habitat,survey_date,geometry,source
25,V0000891695,6542617.0,605553.0,25.03.2017 00:00,B - hea,I - väga hea,B - kõrge väärtus,9050. Rohunditerikkad kuusikud,1.1.6.1. Sinilille kasvukohatüüp,NaN,605553.0,6542617.0,9050. Rohunditerikkad kuusikud,2017-03-25,POINT (605553 6542617),field_plot
2,V0000891672,6568135.0,593705.0,25.03.2017 00:00,B - hea,I - väga hea,A - väga kõrge väärtus,9060. Okasmetsad oosidel ja moreenkuhjatistel,1.1.6.1. Sinilille kasvukohatüüp,NaN,593705.0,6568135.0,9060. Okasmetsad oosidel ja moreenkuhjatistel,2017-03-25,POINT (593705 6568135),field_plot
4,V0000891674,6566846.0,593403.0,25.03.2017 00:00,B - hea,I - väga hea,B - kõrge väärtus,9010*. Vanad loodusmetsad,1.1.4.2. Jänesekapsa kasvukohatüüp,NaN,593403.0,6566846.0,9010*. Vanad loodusmetsad,2017-03-25,POINT (593403 6566846),field_plot
5,V0000891675,6567392.0,593845.0,25.03.2017 00:00,B - hea,I - väga hea,B - kõrge väärtus,91D0*. Siirdesoo- ja rabametsad,1.4.2.1. Siirdesoometsa kasvukohatüüp,NaN,593845.0,6567392.0,91D0*. Siirdesoo- ja rabametsad,2017-03-25,POINT (593845 6567392),field_plot
58,V0000891729,6571514.0,568971.0,25.03.2017 00:00,B - hea,II - hea,B - kõrge väärtus,9050. Rohunditerikkad kuusikud,1.1.6.1. Sinilille kasvukohatüüp,NaN,568971.0,6571514.0,9050. Rohunditerikkad kuusikud,2017-03-25,POINT (568971 6571514),field_plot


In [ ]:
# Extract habitat code into a separate column, keeping the original habitat
plots["habitat_code"] = (plots["habitat"].astype("string").str.strip().str.extract(r"^(?:Eesti\s+)?(0|\d[0-9A-Z]{3}\*?)", flags=re.I, expand=False).str.upper())

In [9]:
print(plots["habitat"].value_counts(dropna=False))

habitat
9010*. Vanad loodusmetsad                                                302
9080. Soostuvad ja soo-lehtmetsad                                        178
0 elupaigatüüp                                                           168
91D0*. Siirdesoo- ja rabametsad                                          152
9050. Rohunditerikkad kuusikud                                           132
9060. Okasmetsad oosidel ja moreenkuhjatistel                            118
91F0. Laialehised lammimetsad                                            114
91E0*. Lammi- ja lodumetsad                                              113
9020*. Vanad laialehised metsad                                          103
9070. Puiskarjamaad                                                       99
2180. Metsastunud luited                                                  98
9180*. Rusukallete ja jäärakute metsad                                    83
6530*. Puisniidud                                                   

In [ ]:
# Define the list of valid forest habitat codes for filtering the plots
codes = ["2180", "9010*", "9020*", "9050", "9060",
         "9080*", "9180*", "91D0*", "91E0*", "91F0*", "9080", "0", "91F0"]

plots = plots[plots["habitat_code"].isin(codes) | plots["habitat_code"].isna()].copy()

print(plots["habitat_code"].value_counts(dropna=False))

habitat_code
9010*    302
9080     178
0        168
91D0*    152
9050     132
9060     118
91F0     114
91E0*    113
9020*    103
2180      98
9180*     83
<NA>       4
Name: count, dtype: Int64


In [11]:
plots.head()

,Proovi / vaatluse kood,Sisestatud x L-EST97,Sisestatud y L-EST97,Seireaja algus,Elupaiga esinduslikkus,Elupaiga funktsioonide säilimine,Elupaiga üldine looduskaitseline väärtus,Elupaigatüüp (LD),Kasvukohatüüp (Paal 1997),Koosluse struktuuri säilimine,x,y,habitat,survey_date,geometry,source,habitat_code
25,V0000891695,6542617.0,605553.0,25.03.2017 00:00,B - hea,I - väga hea,B - kõrge väärtus,9050. Rohunditerikkad kuusikud,1.1.6.1. Sinilille kasvukohatüüp,NaN,605553.0,6542617.0,9050. Rohunditerikkad kuusikud,2017-03-25,POINT (605553 6542617),field_plot,9050
2,V0000891672,6568135.0,593705.0,25.03.2017 00:00,B - hea,I - väga hea,A - väga kõrge väärtus,9060. Okasmetsad oosidel ja moreenkuhjatistel,1.1.6.1. Sinilille kasvukohatüüp,NaN,593705.0,6568135.0,9060. Okasmetsad oosidel ja moreenkuhjatistel,2017-03-25,POINT (593705 6568135),field_plot,9060
4,V0000891674,6566846.0,593403.0,25.03.2017 00:00,B - hea,I - väga hea,B - kõrge väärtus,9010*. Vanad loodusmetsad,1.1.4.2. Jänesekapsa kasvukohatüüp,NaN,593403.0,6566846.0,9010*. Vanad loodusmetsad,2017-03-25,POINT (593403 6566846),field_plot,9010*
5,V0000891675,6567392.0,593845.0,25.03.2017 00:00,B - hea,I - väga hea,B - kõrge väärtus,91D0*. Siirdesoo- ja rabametsad,1.4.2.1. Siirdesoometsa kasvukohatüüp,NaN,593845.0,6567392.0,91D0*. Siirdesoo- ja rabametsad,2017-03-25,POINT (593845 6567392),field_plot,91D0*
58,V0000891729,6571514.0,568971.0,25.03.2017 00:00,B - hea,II - hea,B - kõrge väärtus,9050. Rohunditerikkad kuusikud,1.1.6.1. Sinilille kasvukohatüüp,NaN,568971.0,6571514.0,9050. Rohunditerikkad kuusikud,2017-03-25,POINT (568971 6571514),field_plot,9050


In [ ]:
#One row represents a single plot, so we create a buffer of 20 meters around each plot's geometry. That matches the observation radius used in the dataset.
plots_20 = plots.copy()
plots_20["geometry"] = plots_20.geometry.buffer(20)

In [ ]:
#Land and spatial development board's map square layer read in just in case it is needed for raster masks. It was not used in the current workflow but kept for potential future use.
epk2t = gpd.read_file(r"original\epk2T.shp")

In [14]:
epk2t.head()

,NR,NR10000,LENNUAEG,GSD,ALS_AASTAD,ALS_TAVA_1,ALS_TAVA_2,ALS_TAVA_3,ALS_TAVA_4,ALS_MADAL,ALS_METS,geometry
0,374643,44741,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((643000 6374000, 643000 6375000, 6440..."
1,374644,44741,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((644000 6374000, 644000 6375000, 6450..."
2,374645,44742,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((645000 6374000, 645000 6375000, 6460..."
3,374646,44742,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((646000 6374000, 646000 6375000, 6470..."
4,374647,44742,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((647000 6374000, 647000 6375000, 6480..."


In [15]:
# Select intersecting map squares
selected_squares = gpd.sjoin(epk2t,plots_20[["geometry"]],how="inner",predicate="intersects")

# Remove duplicates (a square may intersect multiple plots)
selected_squares = selected_squares.loc[~selected_squares.index.duplicated()].copy()

print("Selected map squares:", len(selected_squares))

Selected map squares: 1290


In [16]:
selected_squares.to_file(OUTPUT/"plots_squares.gpkg",layer="plots_squares",driver="GPKG")

In [17]:
print(plots["habitat_code"].value_counts(dropna=False))

habitat_code
9010*    302
9080     178
0        168
91D0*    152
9050     132
9060     118
91F0     114
91E0*    113
9020*    103
2180      98
9180*     83
<NA>       4
Name: count, dtype: Int64


In [18]:
display(plots[plots["habitat_code"].isna()])

,Proovi / vaatluse kood,Sisestatud x L-EST97,Sisestatud y L-EST97,Seireaja algus,Elupaiga esinduslikkus,Elupaiga funktsioonide säilimine,Elupaiga üldine looduskaitseline väärtus,Elupaigatüüp (LD),Kasvukohatüüp (Paal 1997),Koosluse struktuuri säilimine,x,y,habitat,survey_date,geometry,source,habitat_code
200,V0000994152,6592365.0,548794.0,10.06.2019 00:00,NaN,NaN,NaN,NaN,1.1.4.2. Jänesekapsa kasvukohatüüp,NaN,548794.0,6592365.0,<NA>,2019-06-10,POINT (548794 6592365),field_plot,<NA>
219,V0000994171,6386855.0,643524.0,22.06.2019 00:00,NaN,NaN,NaN,NaN,2.1.4. Pärisaruniitude tüübirühm,NaN,643524.0,6386855.0,<NA>,2019-06-22,POINT (643524 6386855),field_plot,<NA>
278,V0000994230,6515294.0,518586.0,29.06.2019 00:00,NaN,NaN,NaN,NaN,1.1.3.2. Mustika kasvukohatüüp,NaN,518586.0,6515294.0,<NA>,2019-06-29,POINT (518586 6515294),field_plot,<NA>
254,V0000994206,6470321.0,392478.0,10.07.2019 00:00,NaN,NaN,NaN,NaN,1.1.1.2. Kastikuloo kasvukohatüüp,NaN,392478.0,6470321.0,<NA>,2019-07-10,POINT (392478 6470321),field_plot,<NA>


In [ ]:
#Added Natura 2000 0-habitat layer for further analysis. This data was read in to provide additional habitat information and analyse the possible use of this layer in conjunction with the observation plots.
natura0_elupaik = gpd.read_file("original\\0_elupaik.TAB")

c:\Users\kaursa40847\keskkonnad\andmeteadus314\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: One or several characters couldn't be converted correctly from CP1257 to UTF-8.  This warning will not be emitted anymore
  return ogr_read(


In [20]:
natura0_elupaik.head()

,ID,ELUPAIGATYYP,KKT_PAAL_97,PINDALA,Ala_nimi,ESINDUSLIK,STR_SAIL,FUNKT_SAIL,TAASTATAVUS,LK_SEISUND,ULDINE_LK,MARKUSED,INVENTUUR,INVENTEERIJA,Parandused,kaardistusalus,geometry
0,49,0,,3.23,Kurimetsa,,,,,,,poolkult. heinamaa;,Natura_2002,Pihud,,,"POLYGON ((598718.5 6438897.977, 598814.989 643..."
1,51,0,,12.07,Parmu,,,,,,,"raie vähemal määral, 30a kuusekultuur",Natura_2002,"Ü. Reier, E.Leibak, B.Holm",,,"POLYGON ((696563.967 6387755.587, 696556.353 6..."
2,52,0,,3.75,Parmu,,,,,,,kultuur,Natura_2002,"Ü. Reier, E.Leibak, B.Holm",,,"POLYGON ((696934.365 6388141.045, 697033.998 6..."
3,54,0,,1.34,Mõisametsa,,,,,,,valikraidega ala;istut kuusik,Natura_2002,"Kai Vellak, Ain Vellak",,,"POLYGON ((657936.327 6384304.001, 658039.436 6..."
4,61,0,,3.29,Taarikõnnu,,,,,,,tugevalt harvendatud segamets,Natura_2002,,,,"POLYGON ((552810.34 6511633.839, 552801.568 65..."


In [21]:
natura0_elupaik.dtypes

ID                   int32
ELUPAIGATYYP           str
KKT_PAAL_97            str
PINDALA            float64
Ala_nimi               str
ESINDUSLIK             str
STR_SAIL               str
FUNKT_SAIL             str
TAASTATAVUS            str
LK_SEISUND             str
ULDINE_LK              str
MARKUSED               str
INVENTUUR              str
INVENTEERIJA           str
Parandused             str
kaardistusalus         str
geometry          geometry
dtype: object

In [ ]:
# Filter Natura 2000 0-habitat layer to include only forest habitats and recent inventory years (2017-2025)
natura0_habitats_forests = natura0_elupaik[natura0_elupaik["KKT_PAAL_97"].str.startswith("1", na=False) & natura0_elupaik["INVENTUUR"].astype("string").str.contains(r"201[7-9]|202[0-5]", na=False)].copy()

In [23]:
len(natura0_habitats_forests)

1340

In [24]:
natura0_habitats_forests.crs

<Projected CRS: EPSG:3301>
Name: Estonian Coordinate System of 1997
Axis Info [cartesian]:
- X[north]: Northing (metre)
- Y[east]: Easting (metre)
Area of Use:
- name: Estonia - onshore and offshore.
- bounds: (20.37, 57.52, 28.2, 60.0)
Coordinate Operation:
- name: Estonian National Grid
- method: Lambert Conic Conformal (2SP)
Datum: Estonia 1997
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [ ]:
#Keep only rows with valid dates
dates = pd.to_datetime(natura0_habitats_forests["INVENTUUR"],format="%d.%m.%Y",errors="coerce")


In [26]:
natura0_habitats_forests = natura0_habitats_forests[dates.dt.year.between(2017, 2025)].copy()

In [27]:
len(natura0_habitats_forests)

1139

In [28]:
natura0_habitats_forests.to_file(OUTPUT/"natura0_habitats_forests.gpkg",layer="natura0_habitats_forests",driver="GPKG")

In [ ]:
#Additionally, read in the Natura 2000 forest habitats inventory layer for further analysis. This dataset  could serve as an additional data source..
natura_habitats_forests = gpd.read_file(r"working\natura_habitatsPolygon_forests.gpkg")

In [30]:
natura_habitats_forests.crs

<Projected CRS: EPSG:3301>
Name: Estonian Coordinate System of 1997
Axis Info [cartesian]:
- X[north]: Northing (metre)
- Y[east]: Easting (metre)
Area of Use:
- name: Estonia - onshore and offshore.
- bounds: (20.37, 57.52, 28.2, 60.0)
Coordinate Operation:
- name: Estonian National Grid
- method: Lambert Conic Conformal (2SP)
Datum: Estonia 1997
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [31]:
natura_habitats_forests.head()

,sys_id,versioon,id,pohityyp,pohityyp_p,andmestik,kr_kood,tyybid,tyybid_p,paal_97,...,taastatavs,lk_seisund,uldine_lk,markused,inventisik,invent_kpv,kaitst_ala,natura_ala,area_ha,geometry
0,92229,1.747779e+12,150493087,9080*,90.0,N,NaN,91E0*,10,"1312,1412",...,I,B,B,"keskealised sooviku-lehtmetsad, looduslik stru...",Indrek Tammek�nd,2023-08-01,Alam-Pedja looduskaitseala (KLO1000455),"Alam-Pedja linnuala (RAH0000123),Alam-Pedja lo...",7.611787,"MULTIPOLYGON (((624960.343 6489675.223, 624984..."
1,92212,1.747779e+12,148256581,91D0*,100.0,N,NaN,NaN,NaN,"1421,1511",...,III,C,A,"raiej�rgne, t��piline k�psev monotoonne m�nnik...","Anneli Palo,Indrek Tammek�nd",2023-08-01,Alam-Pedja looduskaitseala (KLO1000455),"Alam-Pedja linnuala (RAH0000123),Alam-Pedja lo...",23.397236,"MULTIPOLYGON (((623834.053 6486740.02, 623916...."
2,83374,1.747779e+12,-14370496,9080*,100.0,N,NaN,NaN,NaN,1312,...,II,B,B,"60-80a sooviku-lehtmetsad, loodusliku arengu j...",Renno Nellis,2021-11-27,Laidevahe looduskaitseala (KLO1000512),Siiksaare-Oessaare lahtede linnuala (RAH000010...,11.947797,"MULTIPOLYGON (((431501.278 6463147.661, 431539..."
3,78650,1.752276e+12,-13363278,9010*,94.0,N,NaN,9080*;91D0*,5;1,1511,...,I,B,B,"tugev kuivendusm�ju, enamasti k�dusoo, ehkki l...",Indrek Sell,2019-08-20,Siiraku looduskaitseala (KLO1000573),"Kikepera linnuala (RAH0000118),Siiraku loodusa...",4.485460,"MULTIPOLYGON (((546158.403 6454430.196, 546226..."
4,87033,1.752276e+12,-20421956,9010*,90.0,N,NaN,9080*,10,1142,...,NaN,B,C,"Paiguti vanu sanitaarraiek�nde, kuid valdavalt...",Indrek Hiiesalu,2022-11-07,"Metsaelupaikade looduskaitseala,Metsaelupaikad...","Otep�� linnuala (RAH0000103),Otep�� loodusala ...",1.253742,"MULTIPOLYGON (((641024.991 6439567.528, 641044..."


In [32]:
natura_habitats_forests["pohityyp_p"].unique()

array([ 90., 100.,  94.,  95.,  50.,  98.,  99.,  80.,  70.,  75.,  97.,
        60.,  85.,  65.,  93.,  88.,  84.,  96.,  30.,  79.,  55.,  25.,
        69.,  59.,  92.,  34.,  89.,  nan,  51.,  52.])

In [33]:
natura_habitats_forests.dtypes

sys_id                 int32
versioon             float64
id                     int32
pohityyp                 str
pohityyp_p           float64
andmestik                str
kr_kood                  str
tyybid                   str
tyybid_p                 str
paal_97                  str
esinduslik               str
str_sail                 str
funkt_sail               str
taastatavs               str
lk_seisund               str
uldine_lk                str
markused                 str
inventisik               str
invent_kpv    datetime64[ms]
kaitst_ala               str
natura_ala               str
area_ha              float64
geometry            geometry
dtype: object

In [ ]:
#Join Natura 2000 0-habitat and forest habitats datasets for further analysis possible use them as same dataset because they are both mapped in similar inventory projects.
natura0 = natura0_habitats_forests.copy()
natura = natura_habitats_forests.copy()

natura0["source"] = "natura0"
natura["source"] = "natura"

# 2. Rename Natura0 columns to match Natura columns
rename_cols = {
    "ELUPAIGATYYP": "pohityyp",
    "KKT_PAAL_97": "paal_97",
    "ESINDUSLIK": "esinduslik",
    "STR_SAIL": "str_sail",
    "FUNKT_SAIL": "funkt_sail",
    "TAASTATAVUS": "taastatavs",
    "LK_SEISUND": "lk_seisund",
    "ULDINE_LK": "uldine_lk",
    "MARKUSED": "markused",
    "INVENTUUR": "invent_kpv",
    "INVENTEERIJA": "inventisik",
}

natura0 = natura0.rename(columns=rename_cols)

# 3. Convert inventory dates to the same type
natura0["invent_kpv"] = pd.to_datetime(natura0["invent_kpv"],format="%d.%m.%Y",errors="coerce")

natura["invent_kpv"] = pd.to_datetime(natura["invent_kpv"],errors="coerce")

# 4. Define columns to retain
keep_cols = [
    "source",
    "id",
    "versioon",
    "pohityyp",
    "pohityyp_p",
    "tyybid",
    "tyybid_p",
    "paal_97",
    "esinduslik",
    "str_sail",
    "funkt_sail",
    "taastatavs",
    "lk_seisund",
    "uldine_lk",
    "markused",
    "inventisik",
    "invent_kpv",
    "kaitst_ala",
    "natura_ala",
    "area_ha",
    "geometry",
    "ID",
    "sys_id"
]

# 5. Ensure both datasets have the same CRS
natura0 = natura0.to_crs(natura.crs)

# 6. Merge row-wise
combined = gpd.GeoDataFrame(pd.concat([natura.reindex(columns=keep_cols), natura0.reindex(columns=keep_cols)], ignore_index=True), geometry="geometry", crs=natura.crs)

# 7. Check results
print("Natura rows:", len(natura))
print("Natura0 rows:", len(natura0))
print("Combined rows:", len(combined))

display(combined.groupby("source").size())
display(combined.head())






Natura rows: 20537
Natura0 rows: 1139
Combined rows: 21676


source
natura     20537
natura0     1139
dtype: int64

,source,id,versioon,pohityyp,pohityyp_p,tyybid,tyybid_p,paal_97,esinduslik,str_sail,...,uldine_lk,markused,inventisik,invent_kpv,kaitst_ala,natura_ala,area_ha,geometry,ID,sys_id
0,natura,150493087.0,1.747779e+12,9080*,90.0,91E0*,10,"1312,1412",C,III,...,B,"keskealised sooviku-lehtmetsad, looduslik stru...",Indrek Tammek�nd,2023-08-01,Alam-Pedja looduskaitseala (KLO1000455),"Alam-Pedja linnuala (RAH0000123),Alam-Pedja lo...",7.611787,"MULTIPOLYGON (((624960.343 6489675.223, 624984...",NaN,92229.0
1,natura,148256581.0,1.747779e+12,91D0*,100.0,NaN,NaN,"1421,1511",C,III,...,A,"raiej�rgne, t��piline k�psev monotoonne m�nnik...","Anneli Palo,Indrek Tammek�nd",2023-08-01,Alam-Pedja looduskaitseala (KLO1000455),"Alam-Pedja linnuala (RAH0000123),Alam-Pedja lo...",23.397236,"MULTIPOLYGON (((623834.053 6486740.02, 623916....",NaN,92212.0
2,natura,-14370496.0,1.747779e+12,9080*,100.0,NaN,NaN,1312,C,II,...,B,"60-80a sooviku-lehtmetsad, loodusliku arengu j...",Renno Nellis,2021-11-27,Laidevahe looduskaitseala (KLO1000512),Siiksaare-Oessaare lahtede linnuala (RAH000010...,11.947797,"MULTIPOLYGON (((431501.278 6463147.661, 431539...",NaN,83374.0
3,natura,-13363278.0,1.752276e+12,9010*,94.0,9080*;91D0*,5;1,1511,C,III,...,B,"tugev kuivendusm�ju, enamasti k�dusoo, ehkki l...",Indrek Sell,2019-08-20,Siiraku looduskaitseala (KLO1000573),"Kikepera linnuala (RAH0000118),Siiraku loodusa...",4.485460,"MULTIPOLYGON (((546158.403 6454430.196, 546226...",NaN,78650.0
4,natura,-20421956.0,1.752276e+12,9010*,90.0,9080*,10,1142,C,II,...,C,"Paiguti vanu sanitaarraiek�nde, kuid valdavalt...",Indrek Hiiesalu,2022-11-07,"Metsaelupaikade looduskaitseala,Metsaelupaikad...","Otep�� linnuala (RAH0000103),Otep�� loodusala ...",1.253742,"MULTIPOLYGON (((641024.991 6439567.528, 641044...",NaN,87033.0


In [35]:
combined.dtypes

source                   str
id                   float64
versioon             float64
pohityyp                 str
pohityyp_p           float64
tyybid                object
tyybid_p              object
paal_97                  str
esinduslik               str
str_sail                 str
funkt_sail               str
taastatavs               str
lk_seisund               str
uldine_lk                str
markused                 str
inventisik               str
invent_kpv    datetime64[us]
kaitst_ala            object
natura_ala            object
area_ha              float64
geometry            geometry
ID                   float64
sys_id               float64
dtype: object

In [36]:
combined["invent_kpv"].unique()

<DatetimeArray>
['2023-08-01 00:00:00', '2021-11-27 00:00:00', '2019-08-20 00:00:00',
 '2022-11-07 00:00:00', '2020-04-04 00:00:00', '2022-04-30 00:00:00',
 '2020-05-05 00:00:00', '2018-04-22 00:00:00', '2020-04-19 00:00:00',
 '2025-11-05 00:00:00',
 ...
 '2024-01-18 00:00:00', '2025-01-10 00:00:00', '2021-07-04 00:00:00',
 '2025-11-13 00:00:00', '2025-11-21 00:00:00', '2023-06-19 00:00:00',
 '2025-12-04 00:00:00', '2025-08-07 00:00:00', '2025-06-12 00:00:00',
 '2025-12-06 00:00:00']
Length: 1628, dtype: datetime64[us]

In [ ]:
# Filter observation plots to include only those surveyed after June 2017. Since the oldest Sentinel imagery used for analysis is from mid-2017, earlier plots might not have corresponding imagery available.
plots_after_june_2017 = plots[plots["survey_date"] >= "2017-06-01"]
print(len(plots_after_june_2017))
plots_20_after_june_2017 = plots_20[plots_20["survey_date"] >= "2017-06-01"]
print(len(plots_20_after_june_2017))

1553
1553


In [38]:
plots_after_june_2017.to_file(OUTPUT/"observation_plots_Natura.gpkg",layer="observation_plots_Natura",driver="GPKG")
plots_20_after_june_2017.to_file(OUTPUT/"observation_plots_Natura_20.gpkg",layer="observation_plots_Natura_20",driver="GPKG")

In [39]:
plots.dtypes

Proovi / vaatluse kood                                 str
Sisestatud x L-EST97                                   str
Sisestatud y L-EST97                                   str
Seireaja algus                                         str
Elupaiga esinduslikkus                                 str
Elupaiga funktsioonide säilimine                       str
Elupaiga üldine looduskaitseline väärtus               str
Elupaigatüüp (LD)                                      str
Kasvukohatüüp (Paal 1997)                              str
Koosluse struktuuri säilimine                          str
x                                                  float64
y                                                  float64
habitat                                             string
survey_date                                 datetime64[us]
geometry                                          geometry
source                                                 str
habitat_code                                        stri

In [ ]:
#Final reference dataset for reference was chosen to be the observation plots surveyed after June 2017. Inventory polygons from Natura 2000 layers are not included in this final reference dataset but could be used for method development later on.
# ndvi_plots was named first to test raster data extraction from Sentinel imagery. It was renamed later on as final reference dataset.
ndvi_plots = plots_20_after_june_2017.copy().reset_index(drop=True)
ndvi_plots["plot_id"] = ndvi_plots.index

ndvi_plots["survey_date"] = pd.to_datetime(ndvi_plots["survey_date"], errors="coerce")


In [ ]:
# Raster year adjustment based on survey date. Earlier image than observation is preferred, because it better represents the conditions at the time of the survey. Later images might  incorporate bigger land cover changes that occurred after the survey.
ndvi_plots["ndvi_year"] = ndvi_plots["survey_date"].dt.year

ndvi_plots.loc[(ndvi_plots["survey_date"].dt.month <= 8) &(ndvi_plots["survey_date"].dt.year > 2017),"ndvi_year"] -= 1

In [42]:
ndvi_plots.head()

,Proovi / vaatluse kood,Sisestatud x L-EST97,Sisestatud y L-EST97,Seireaja algus,Elupaiga esinduslikkus,Elupaiga funktsioonide säilimine,Elupaiga üldine looduskaitseline väärtus,Elupaigatüüp (LD),Kasvukohatüüp (Paal 1997),Koosluse struktuuri säilimine,x,y,habitat,survey_date,geometry,source,habitat_code,plot_id,ndvi_year
0,V0000891786,6397087.0,648523.0,01.06.2017 00:00,A - väga hea,I - väga hea,A - väga kõrge väärtus,9080. Soostuvad ja soo-lehtmetsad,1.5.1.2. Jänesekapsa-kõdusoo kasvukohatüüp,NaN,648523.0,6397087.0,9080. Soostuvad ja soo-lehtmetsad,2017-06-01,"POLYGON ((648543 6397087, 648542.904 6397085.0...",field_plot,9080,0,2017
1,V0000891788,6398159.0,648833.0,01.06.2017 00:00,B - hea,I - väga hea,A - väga kõrge väärtus,91D0*. Siirdesoo- ja rabametsad,1.4.2.1. Siirdesoometsa kasvukohatüüp,NaN,648833.0,6398159.0,91D0*. Siirdesoo- ja rabametsad,2017-06-01,"POLYGON ((648853 6398159, 648852.904 6398157.0...",field_plot,91D0*,1,2017
2,V0000891736,6383581.0,651414.0,02.06.2017 00:00,C - arvestatav,II - hea,A - väga kõrge väärtus,9180*. Rusukallete ja jäärakute metsad,1.1.6.2. Naadi kasvukohatüüp,NaN,651414.0,6383581.0,9180*. Rusukallete ja jäärakute metsad,2017-06-02,"POLYGON ((651434 6383581, 651433.904 6383579.0...",field_plot,9180*,2,2017
3,V0000891819,6458707.0,389146.0,14.06.2017 00:00,C - arvestatav,I - väga hea,B - kõrge väärtus,9050. Rohunditerikkad kuusikud,1.3.1.3. Sinihelmika kasvukohatüüp,NaN,389146.0,6458707.0,9050. Rohunditerikkad kuusikud,2017-06-14,"POLYGON ((389166 6458707, 389165.904 6458705.0...",field_plot,9050,3,2017
4,V0000891817,6458791.0,389869.0,14.06.2017 00:00,A - väga hea,II - hea,A - väga kõrge väärtus,9060. Okasmetsad oosidel ja moreenkuhjatistel,1.1.5.1. Kassikäpa kasvukohatüüp,NaN,389869.0,6458791.0,9060. Okasmetsad oosidel ja moreenkuhjatistel,2017-06-14,"POLYGON ((389889 6458791, 389888.904 6458789.0...",field_plot,9060,4,2017


In [ ]:
#Latest Sentinel imagery adjustment for plots surveyed after September 2025. For these plots, the available remote sensing year is set to 2025, because newer Sentinel imagery is not available on geokuup.ee platform
mask = ndvi_plots["survey_date"] >= "2025-09-01"

ndvi_plots.loc[mask, "ndvi_year"] = 2025

ndvi_plots = ndvi_plots.rename(columns={"ndvi_year": "available_remote_sensing_year"})

In [44]:
epk2t.head()

,NR,NR10000,LENNUAEG,GSD,ALS_AASTAD,ALS_TAVA_1,ALS_TAVA_2,ALS_TAVA_3,ALS_TAVA_4,ALS_MADAL,ALS_METS,geometry
0,374643,44741,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((643000 6374000, 643000 6375000, 6440..."
1,374644,44741,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((644000 6374000, 644000 6375000, 6450..."
2,374645,44742,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((645000 6374000, 645000 6375000, 6460..."
3,374646,44742,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((646000 6374000, 646000 6375000, 6470..."
4,374647,44742,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((647000 6374000, 647000 6375000, 6480..."


In [ ]:
#Final selection of map squares intersecting with observation plots for the available remote sensing years. Tiles were not used at this stage, but could be useful for further spatial analysis.
tiles = epk2t.copy()

selected_squares_RS = gpd.sjoin(tiles,ndvi_plots[["available_remote_sensing_year", "geometry"]],how="inner",predicate="intersects")

# Keep one square per NDVI year
selected_squares_RS = selected_squares_RS.drop_duplicates(subset=["NR", "available_remote_sensing_year"]).drop(columns="index_right")

# Save as spatial layer
selected_squares_RS.to_file(OUTPUT / "map_squares_query.gpkg",layer="map_squares_query",driver="GPKG")

print("Required tile-year combinations:", len(selected_squares_RS))

Required tile-year combinations: 1495


In [46]:
ndvi_plots.to_file("working\\obsevation_plots_with_year.gpkg", driver="GPKG")

In [47]:
print(ndvi_plots[ndvi_plots["survey_date"]>="2025-09-01"])

     Proovi / vaatluse kood Sisestatud x L-EST97 Sisestatud y L-EST97  \
1314            V0001497652    6491081.575237704     627855.483653559   
1315            V0001497704    6490779.215104329    627582.1925049603   
1316            V0001497716    6490942.735347615    628893.0124496839   
1317            V0001497723    6491580.313595575    629177.0107090938   
1318            V0001497592    6491631.529726475      629575.73076364   
...                     ...                  ...                  ...   
1548            V0001603862    6544050.066675108    522385.2883214021   
1549            V0001603980    6560187.238331622    697718.9870252868   
1550            V0001603842    6421520.834672567    605588.1708961141   
1551            V0001603931    6417854.809341499    610503.9000632523   
1552            V0001604028    6421707.344369157    604581.1617628322   

        Seireaja algus Elupaiga esinduslikkus  \
1314  03.09.2025 00:00                B - hea   
1315  03.09.2025 00:00   